In [ ]:
# Demo 5: Test claim extraction with cleaned/chunked transcript (chunk batching)

# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api

In [ ]:
# Libraries
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from transformers import AutoTokenizer

# Integrate youtube_api.py code:
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# Load tokenizer with left padding for Qwen
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", padding_side ='left')



In [ ]:

# Integrate youtube_api.py code:

load_dotenv()

api_key= os.getenv("API_KEY")

yta = YouTubeTranscriptApi()

"""
def get_transcript(video_id):
    try:
        transcript = yta.fetch(video_id)
        transcript_text = " ".join(entry.text for entry in transcript)
        return transcript_text
    except Exception:
        return None
"""

youtube = build('youtube', 'v3', developerKey=api_key)

request = youtube.videos().list(
    part='localizations,statistics,topicDetails',
    chart='mostPopular',
    regionCode='us',
    videoCategoryId='20',
    maxResults='50'
)

response = request.execute()

videos_data = []

for item in response["items"]:
    video_id = item.get("id")

    videos_data.append({"Video ID": video_id})

df = pd.DataFrame(videos_data)
# df["Transcript"] = df["Video ID"].apply(get_transcript)


In [ ]:
# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    tokenizer = tokenizer,
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=200 # increase to prevent incomplete output
)

In [6]:
# Test clean raw transcript

def clean_transcript(transcript):
  cleaned = [] # list for cleaned snippets

  # Loop to get text (clean) & get start time
  for snippet in transcript.snippets:
    text = snippet.text # get text from snippet
    start_time = snippet.start # get start time from snippet

    while "[" in text:
      start = text.index("[")
      end = text.index("]") + 1
      caption = text[start:end]
      text = text.replace(caption, "")

    cleaned.append({"text": text, "start": start_time}) # add snippet's text and start time to cleaned list

  # Test print cleaned list
  # print(f"Transcript Text:\n{cleaned}")

  return cleaned

In [7]:
# Test chunking cleaned transcript

def chunk_transcript(cleaned):
  chunk_size = 300 # chunk size based on time (sec)
  chunks = [] # list for chunks
  chunk_snippets = [] # list to build each chunk
  curr_chunk_time = 0

  # Loop through all snippets in cleaned[]
  for snippet in cleaned:
      text = snippet["text"] # get snippet text
      chunk_snippets.append(text) # add text to chunk_snippets list

      # Check if reach chunk_size (approx.)
      if snippet["start"] - curr_chunk_time >= chunk_size:
        # print(f"Chunk time: {snippet["start"] - curr_chunk_time} sec") # Test print chunk start time
        chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
        chunks.append(chunk) # add chunk to list of chunks

        chunk_snippets = [] # reset chunk_snippets list
        curr_chunk_time = snippet["start"] # reset time to current start time

  # If leftover snippets, create and add last chunk
  if chunk_snippets:
    chunk = " ".join(chunk_snippets)  # join chunk snippet texts into a string
    chunks.append(chunk) # add chunk to list of chunks

  # Test print chunks list
  # print(f"Transcript Text:\n{chunks}")

  return chunks


In [8]:
# Test cleaned/chunked transcript in claim extraction step (chunk batching)

# Function to extract claims
def claim_extraction(chunks):
  message_list = []
  # Create message more each chunk
  for chunk in chunks:
    messages = [
      {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
      {"role": "user", "content": f"Extract all factual claims as bullet points.\n\nText: {chunk}"}
    ]
    message_list.append(messages) # add message to list

  results = []
  # Send messages to Qwen model (process in batches)
  for output in pipe(message_list, batch_size=5, max_length=None):
    # Get claim from Qwen response
    result = output[0]["generated_text"][-1]["content"]
    results.append(result) # add qwen output to list
  return results


In [ ]:
# Test extract claim from multiple videos

# json library
import json

# Short videos sample list test:
# video_ids = ["-1Sy6iz43vg", "TT4HTDYhiOU"] # list of short sample video ids

# 50 video ids from df into a list test:
video_ids = df["Video ID"].tolist()

video_claims = []
# Loop through each video id in list
for video_id in video_ids:
  yta = YouTubeTranscriptApi()
  try:
    transcript = yta.fetch(video_id) # get transcript
    chunks = chunk_transcript(clean_transcript(transcript)) # clean/chunk transcript

    # Extract claims from chunks list
    claims_list = claim_extraction(chunks)

    video_claims.append({"video_id": video_id, "claims": claims_list})

    # Print claims
    """
    for claims in claims_list:
      print(f"Claims Extracted:\n{claims}\n")
    """
  except Exception: # error for no transcript
    print(f"No transcript for video ID: {video_id}")

with open('video_claims.json', 'w') as file:
  json.dump(video_claims, file, indent=4)



Short videos sample list:

First video: (6:15 min, 60 sec chunks, batch size = 3, 3 batches)

Claims Extracted:
- My little brother recently got Minecraft.
- The learning curve for Minecraft is steep.
- There's no clear path provided by the game itself.
- One needs to figure things out themselves.
- The video aims to teach basic Minecraft skills.
- It helps new players understand the game's basics.
- Players should gather enough wood (at least 32) initially.
- Gathering wood involves cutting down trees using hands or an axe.
- Using axes instead of hands saves time during gathering.
- A crafting table is essential for creating tools like a wooden pickaxe.
- Digging below blocks is necessary after obtaining the tools.
- The process starts with setting up, focusing on gathering wood.
- Starting with 32 logs ensures there's sufficient material.
- Crafting tables facilitate tool creation efficiently.
- Wooden picks are created using planks from harvested logs.
- This guide aims to assist beginners effectively.

Claims Extracted:
- You need to collect six logs to make a Stone Axe.
- The Stone Axe will allow you to gather more wood efficiently.
- Use the Stone Axe to obtain a full stack of logs, which will last for a while.
- Craft a Stone Pickaxe using the additional three stones collected.
- Complete Step Two by either finding animals (preferably cows) to get meat or locating a nearby Village to collect food items like bread or hay.
- Keep an eye on the sun's position when completing Step Three; avoid doing so after sunset as it becomes much harder.

Claims Extracted:
- You can craft a bed by placing down three wool and three planks.
- Zombies can break down wooden doors, so it's important to place a block in front of them to prevent monsters from entering.
- If you found a village, you can hide inside one of the buildings for safety.
- To find iron, you can collect it from Iron Golems or look for chests in villages.
- Once you've collected enough iron, you should proceed to step four.
- Step four involves finding iron and collecting materials like coal and iron ore to craft torches.

Claims Extracted:
- Use sticks and coal to obtain raw iron.
- Gather eight more pieces of stone, break them, and use the cobblestone to craft a furnace.
- Place raw iron and coal in the furnace to smelt it into iron ingots.
- Craft a shield using one iron ingot and five wooden planks.
- Build a basic shelter using a simple house design with a bed, crafting table, chests, and furnaces.
- Protect the house by placing a door and torches around the perimeter to prevent zombie attacks.

Claims Extracted:
- Use three pieces of iron to craft a bucket.
- Find water by finding stones or sticks outdoors.
- Craft a stone hoe with two pieces of stone and two sticks.
- Break one block, fill it with water, and use the hoe to plow the land.
- Plant seeds from short grass on farmland for food.
- Collect seeds from short grass and plant them on farmland.
- Venturing deeper into caves to locate more resources.
- Collect more iron, coal, and look for diamonds.
- Create a fence to contain cows and sheep near the base.
- Breed cows and sheep using wheat from the wheat farm.
- Keep the wheat in hand so the animals will follow.

Claims Extracted:
- You can feed baby zombies wheat, which will allow them to grow into adult zombies.
- Adult zombies can breed, producing baby zombies.
- Saplings can be harvested from trees nearby and planted near the base, making it easier to gather wood later.
- Logs can be collected more efficiently by planting saplings instead of traveling far away.
- Obi obsidian can be crafted using sugar cane, leather, and two diamonds.
- Books can be enchanted by placing an Enchanting Table and surrounding it with Bookshelves.
- Lapis Lazuli is found in caves and used to enchant equipment.
- Experience Points (XP) can be earned through mining ores or hunting monsters.
- Enchantments can increase the power of weapons and armor.
- Building a larger base requires careful planning based on the given tutorials.

Claims Extracted:
- You want to see a part 2 of this.
- Please let me know in the comments.
- I hope it helped you.
- Subscribe for more videos similar to this.


Second video: (4:16 min, 60 sec chunks, batch size = 3, 2 batches)


Claims Extracted:
- Minecraft 26.0 update includes over 50 changes.
- Menu layout redesign: Settings button moved back to top for natural feel; improved design and navigation.
- Subtitles added to show sounds as text on screen, enhancing visibility and awareness during gameplay.
- Beta status for these features before full release.

Claims Extracted:
- Bamboo fades out or disappears suddenly when moved away.
- Rendering issue with bamboo has been fixed.
- Zombie horses now stay calm even while burning due to a bug fix.
- Baby horses now grow instantly instead of slowly.
- Bedrock and Java no longer have different textures for the same block.
- Drowned mobs no longer hold nautilus shells, tridents, and fishing rods.
- Diamond horse armor now shows its actual armor toughness correctly.

Claims Extracted:
- Spawn eggs do not spawn baby mobs in Bedrock.
- Squid babies have improved textures with smoother and more detailed appearance.
- Adult polar bears cannot be killed by hitting them.
- Nautilus swimming animation is now more natural with better sound effects.
- Zombie horse and nautilus damage issues have been fixed.
- Johnny feature works correctly in Bedrock.

Claims Extracted:
- Switching modes is made faster and easier.
- Paper and nugget crafting in the crafting table is added.
- Baby zombies, baby husks, and baby drowned have slightly larger heads making them easier to hit.
- Baby piglins and zombified piglins in the Nether look cuter.
- Baby villagers and zombie villagers have smaller noses.
- A new party system is introduced under the social tab.
- The party system allows controlling who can send invitations, inviting only friends, or setting the world to be open for direct invitations.
- There's an intention to add voice chat based on the new system.

Claims Extracted:
- The update definitely improves the game's smoothness.
- Share videos of the update with friends if you enjoy it.
- Leave a comment to let me know which feature is your favorite.
- See you in the next video.

